# 01. DART 데이터 수집

이 노트북은 `02_dart_data_collection.md` 지침에 맞춰 OpenDART API를 직접 호출해 사업보고서 식별자와 ESG passage 후보를 수집합니다.

분석 단위는 회사명이 아니라 `stock_code × fiscal_year`입니다. `data/raw/company_master.csv`의 381개 기업-연도에서 시작해 KCGS 평가연도(`esg_year`)와 직전 회계연도(`fiscal_year`) 사업보고서를 연결합니다.

생성 산출물은 세 가지입니다.

- `data/processed/dart_collection_checkpoint.csv`: 381행 firm-year 단위 체크포인트 CSV
- `data/processed/dart_passages.csv`: passage 단위 분석용 CSV
- `data/processed/dart_reports/{stock_code}_{fiscal_year}.json`: firm-year별 원문 섹션, 식별자, passage 후보, 실패 사유 캐시

기본 실행 흐름은 삼성전자 샘플 1건까지입니다. 전체 381행 수집은 마지막 배치 실행 셀을 명시적으로 실행하세요.

## 1. 설정, 경로, API 키 로드

`.env`, `DART_API_KEY`, `OPENDART_API_KEY` 순서로 API 키를 읽습니다. 노트북 실행 위치가 프로젝트 루트가 아니어도 `data/` 또는 `pyproject.toml`을 기준으로 루트를 찾습니다.

In [1]:
import io
import json
import os
import re
import time
import zipfile
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from html import unescape
from pathlib import Path
from typing import Any

import pandas as pd
import requests


def find_project_root() -> Path:
    """현재 작업 위치에서 프로젝트 루트를 찾습니다."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'pyproject.toml').exists() or (candidate / 'data').exists():
            return candidate
    return Path.cwd()


def load_env_file(env_path: Path) -> None:
    """간단한 KEY=VALUE 형식의 .env 파일을 환경변수로 읽습니다."""
    if not env_path.exists():
        return
    for raw_line in env_path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


PROJECT_ROOT = find_project_root()
load_env_file(PROJECT_ROOT / '.env')

API_KEY = os.getenv('DART_API_KEY') or os.getenv('OPENDART_API_KEY')
if not API_KEY:
    raise RuntimeError('OpenDART API 키가 없습니다. .env에 DART_API_KEY 또는 OPENDART_API_KEY를 설정하세요.')

OPENDART_BASE = 'https://opendart.fss.or.kr/api'
REQUEST_TIMEOUT = 60
SLEEP_BETWEEN_CALLS = 0.30

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROCESSED_DIR / 'dart_reports'

COMPANY_MASTER_PATH = RAW_DIR / 'company_master.csv'
SEED_DICTIONARY_PATH = RAW_DIR / 'seed_dictionary.csv'
CORP_CODE_PATH = RAW_DIR / 'CORPCODE.xml'
CHECKPOINT_CSV_PATH = PROCESSED_DIR / 'dart_collection_checkpoint.csv'
PASSAGES_CSV_PATH = PROCESSED_DIR / 'dart_passages.csv'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'CHECKPOINT_CSV_PATH = {CHECKPOINT_CSV_PATH}')
print(f'PASSAGES_CSV_PATH = {PASSAGES_CSV_PATH}')

PROJECT_ROOT = /Users/home1/Documents/esg-dart-project
CHECKPOINT_CSV_PATH = /Users/home1/Documents/esg-dart-project/data/processed/dart_collection_checkpoint.csv
PASSAGES_CSV_PATH = /Users/home1/Documents/esg-dart-project/data/processed/dart_passages.csv


## 2. `company_master.csv` 입력 검증

`company_master.csv`는 381행이어야 하며, `stock_code × fiscal_year`가 고유해야 합니다. `stock_code`는 숫자로 읽히면 앞의 0이 사라지므로 항상 6자리 문자열로 보정합니다.

In [2]:
def load_company_master(path: Path = COMPANY_MASTER_PATH) -> pd.DataFrame:
    """company_master.csv를 읽고 분석 키를 정규화합니다."""
    df = pd.read_csv(
        path,
        dtype={
            'stock_code': 'string',
            'corp_code': 'string',
            'rcept_no': 'string',
            'report_code': 'string',
        },
    )
    df = df.copy()
    df['stock_code'] = df['stock_code'].str.replace(r'\.0$', '', regex=True).str.zfill(6)
    df['fiscal_year'] = df['fiscal_year'].astype(int)
    df['esg_year'] = df['esg_year'].astype(int)
    return df


def validate_company_master(df: pd.DataFrame) -> dict[str, Any]:
    """수집 전 입력 표의 기본 조건을 확인합니다."""
    issues = []

    if len(df) != 381:
        issues.append(f'행 수가 381이 아닙니다: {len(df)}')
    if df['stock_code'].nunique() != 127:
        issues.append(f'고유 stock_code 수가 127이 아닙니다: {df["stock_code"].nunique()}')

    duplicate_count = int(df.duplicated(['stock_code', 'fiscal_year']).sum())
    if duplicate_count:
        issues.append(f'중복 stock_code × fiscal_year 행이 있습니다: {duplicate_count}')

    mismatch_count = int((df['esg_year'] != df['fiscal_year'] + 1).sum())
    if mismatch_count:
        issues.append(f'esg_year != fiscal_year + 1 행이 있습니다: {mismatch_count}')

    year_counts = df.groupby('fiscal_year').size().to_dict()
    for fiscal_year in [2022, 2023, 2024]:
        if year_counts.get(fiscal_year, 0) != 127:
            issues.append(f'{fiscal_year}년 행 수가 127이 아닙니다: {year_counts.get(fiscal_year, 0)}')

    missing_year_sets = (
        df.groupby('stock_code')['fiscal_year']
        .apply(lambda years: sorted(set(years)) != [2022, 2023, 2024])
    )
    bad_stock_count = int(missing_year_sets.sum())
    if bad_stock_count:
        issues.append(f'2022, 2023, 2024 세 연도를 모두 갖지 않는 stock_code가 있습니다: {bad_stock_count}')

    return {
        'rows': len(df),
        'unique_stock_code': int(df['stock_code'].nunique()),
        'fiscal_year_counts': {int(k): int(v) for k, v in year_counts.items()},
        'fiscal_years': sorted(int(x) for x in df['fiscal_year'].unique()),
        'esg_years': sorted(int(x) for x in df['esg_year'].unique()),
        'duplicate_stock_year_rows': duplicate_count,
        'issues': issues,
    }


company_master = load_company_master()
input_report = validate_company_master(company_master)

print(json.dumps(input_report, ensure_ascii=False, indent=2))
company_master.head(10)

{
  "rows": 381,
  "unique_stock_code": 127,
  "fiscal_year_counts": {
    "2022": 127,
    "2023": 127,
    "2024": 127
  },
  "fiscal_years": [
    2022,
    2023,
    2024
  ],
  "esg_years": [
    2023,
    2024,
    2025
  ],
  "duplicate_stock_year_rows": 0,
  "issues": []
}


,company_name,corp_code,stock_code,industry,fiscal_year,report_code,rcept_no,esg_source,esg_grade,e_grade,s_grade,g_grade,esg_year
0,삼성전자,<NA>,005930,전기전자,2022,11011,<NA>,한국ESG기준원,A,A,A+,B+,2023
1,삼성전자,<NA>,005930,전기전자,2023,11011,<NA>,한국ESG기준원,B+,B+,A,B,2024
2,삼성전자,<NA>,005930,전기전자,2024,11011,<NA>,한국ESG기준원,A,B+,A+,B+,2025
3,BYC,<NA>,001460,섬유/의류,2022,11011,<NA>,한국ESG기준원,D,D,D,D,2023
4,BYC,<NA>,001460,섬유/의류,2023,11011,<NA>,한국ESG기준원,D,D,D,C,2024
5,BYC,<NA>,001460,섬유/의류,2024,11011,<NA>,한국ESG기준원,D,D,D,C,2025
6,현대자동차,<NA>,005380,자동차,2022,11011,<NA>,한국ESG기준원,A,A,A,B+,2023
7,현대자동차,<NA>,005380,자동차,2023,11011,<NA>,한국ESG기준원,A,A+,B+,A,2024
8,현대자동차,<NA>,005380,자동차,2024,11011,<NA>,한국ESG기준원,A,A+,B+,B+,2025
9,CJ대한통운,<NA>,000120,물류,2022,11011,<NA>,한국ESG기준원,A,A,A+,B+,2023


## 3. `corp_code` 다운로드, 캐시, 매핑

OpenDART `corpCode.xml`은 전체 회사 코드 ZIP을 반환합니다. 한 번 내려받아 `data/raw/CORPCODE.xml`로 캐시한 뒤, `stock_code`에서 `corp_code`를 찾습니다.

In [3]:
def download_corp_code_xml(force: bool = False) -> Path:
    """OpenDART corpCode.xml을 다운로드하고 캐시합니다."""
    if CORP_CODE_PATH.exists() and not force:
        print(f'캐시 사용: {CORP_CODE_PATH}')
        return CORP_CODE_PATH

    response = requests.get(
        f'{OPENDART_BASE}/corpCode.xml',
        params={'crtfc_key': API_KEY},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()

    if response.content[:2] != b'PK':
        snippet = response.content[:500].decode('utf-8', errors='replace')
        raise RuntimeError(f'corpCode.xml 응답이 ZIP이 아닙니다: {snippet}')

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
        zip_file.extract('CORPCODE.xml', path=CORP_CODE_PATH.parent)

    print(f'저장 완료: {CORP_CODE_PATH} ({CORP_CODE_PATH.stat().st_size:,} bytes)')
    return CORP_CODE_PATH


def build_stock_to_corp_map() -> dict[str, str]:
    """CORPCODE.xml에서 {stock_code: corp_code} 매핑을 만듭니다."""
    if not CORP_CODE_PATH.exists():
        download_corp_code_xml()

    tree = ET.parse(CORP_CODE_PATH)
    root = tree.getroot()
    candidates: dict[str, dict[str, str]] = {}

    for item in root.findall('list'):
        stock_code = (item.findtext('stock_code') or '').strip().zfill(6)
        corp_code = (item.findtext('corp_code') or '').strip()
        modify_date = (item.findtext('modify_date') or '').strip()
        if not stock_code.strip('0') or not corp_code:
            continue
        previous = candidates.get(stock_code)
        if previous is None or modify_date > previous['modify_date']:
            candidates[stock_code] = {'corp_code': corp_code, 'modify_date': modify_date}

    return {stock_code: row['corp_code'] for stock_code, row in candidates.items()}


# 최초 실행 시 네트워크 호출이 발생합니다.
download_corp_code_xml()
stock_to_corp = build_stock_to_corp_map()

missing_corp = sorted(set(company_master['stock_code']) - set(stock_to_corp))
print(f'매핑된 상장 stock_code 수: {len(stock_to_corp):,}')
print(f'company_master 중 corp_code 미매핑: {len(missing_corp)}')
print('삼성전자 005930 ->', stock_to_corp.get('005930'))
if missing_corp:
    print('미매핑 예시:', missing_corp[:10])

저장 완료: /Users/home1/Documents/esg-dart-project/data/raw/CORPCODE.xml (29,896,911 bytes)
매핑된 상장 stock_code 수: 3,965
company_master 중 corp_code 미매핑: 0
삼성전자 005930 -> 00126380


## 4. `list.json` 사업보고서 조회 및 `rcept_no` 선택

`list.json`에서 `fiscal_year + 1` 공시연도의 사업보고서를 조회합니다. 선택 기준은 대상 회계연도가 보고서명에 표시된 사업보고서이며, `[기재정정]`은 원본보다 우선하고 `[첨부정정]`은 후순위로 둡니다.

In [4]:
@dataclass
class ReportSelection:
    report_nm: str
    rcept_no: str
    rcept_dt: str
    corp_code: str
    stock_code: str
    fiscal_year: int

    @property
    def viewer_url(self) -> str:
        return f'https://dart.fss.or.kr/dsaf001/main.do?rcpNo={self.rcept_no}'


def opendart_get_json(path: str, params: dict[str, Any], retries: int = 2) -> dict[str, Any]:
    """OpenDART JSON API를 호출합니다."""
    request_params = {'crtfc_key': API_KEY, **params}
    last_error = None
    for attempt in range(retries + 1):
        try:
            response = requests.get(
                f'{OPENDART_BASE}/{path}',
                params=request_params,
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            return response.json()
        except (requests.RequestException, ValueError) as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'OpenDART {path} 호출 실패: {last_error}')


def list_annual_reports(corp_code: str, fiscal_year: int) -> list[dict[str, Any]]:
    """회계연도 다음 해에 공시된 사업보고서 목록을 조회합니다."""
    disclosure_year = fiscal_year + 1
    payload = opendart_get_json(
        'list.json',
        {
            'corp_code': corp_code,
            'bgn_de': f'{disclosure_year}0101',
            'end_de': f'{disclosure_year}1231',
            'pblntf_detail_ty': 'A001',
            'page_count': '100',
        },
    )

    status = payload.get('status')
    if status in {'013', '014'}:
        return []
    if status != '000':
        raise RuntimeError(f'list.json 에러: {status} {payload.get("message")}')
    return payload.get('list', [])


def report_name_has_target_year(report_name: str, fiscal_year: int) -> bool:
    """보고서명에서 대상 회계연도를 확인합니다. 예: 사업보고서 (2024.12)."""
    compact = re.sub(r'\s+', '', report_name or '')
    return bool(re.search(rf'\({fiscal_year}(?:\.|\))', compact))


def annual_report_priority(row: dict[str, Any], fiscal_year: int) -> tuple[int, str, str]:
    """정정공시와 접수일 기준으로 사업보고서 선택 우선순위를 계산합니다."""
    report_nm = row.get('report_nm', '') or ''
    is_business_report = '사업보고서' in report_nm
    has_target_year = report_name_has_target_year(report_nm, fiscal_year)
    is_amended_text = '[기재정정]' in report_nm
    is_attachment_only = '[첨부정정]' in report_nm

    score = 0
    if is_business_report:
        score += 100
    if has_target_year:
        score += 1000
    if is_amended_text:
        score += 20
    if is_attachment_only:
        score -= 20

    return (score, row.get('rcept_dt', '') or '', row.get('rcept_no', '') or '')


def select_annual_report(corp_code: str, stock_code: str, fiscal_year: int) -> ReportSelection:
    """OpenDART 목록에서 대상 회계연도 사업보고서를 선택합니다."""
    rows = list_annual_reports(corp_code, fiscal_year)
    candidates = [
        row for row in rows
        if '사업보고서' in (row.get('report_nm', '') or '')
        and report_name_has_target_year(row.get('report_nm', '') or '', fiscal_year)
    ]

    if not candidates:
        names = [row.get('report_nm', '') for row in rows[:5]]
        raise LookupError(f'{fiscal_year} 회계연도 사업보고서를 찾지 못했습니다. 조회 예시: {names}')

    selected = max(candidates, key=lambda row: annual_report_priority(row, fiscal_year))
    return ReportSelection(
        report_nm=selected.get('report_nm', '') or '',
        rcept_no=selected.get('rcept_no', '') or '',
        rcept_dt=selected.get('rcept_dt', '') or '',
        corp_code=corp_code,
        stock_code=stock_code,
        fiscal_year=fiscal_year,
    )

## 5. `document.xml` 원문 다운로드

`document.xml`은 ZIP 파일을 반환합니다. ZIP 안의 XML 파일을 모두 읽고, 인코딩 차이를 고려해 문자열로 변환합니다.

In [5]:
def decode_xml_bytes(raw: bytes) -> str:
    """DART XML에서 자주 보이는 인코딩을 순서대로 시도합니다."""
    for encoding in ('utf-8', 'euc-kr', 'cp949'):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode('utf-8', errors='replace')


def fetch_report_xml_files(rcept_no: str) -> list[tuple[str, str]]:
    """document.xml API에서 보고서 ZIP을 받아 XML 텍스트 목록으로 반환합니다."""
    last_error = None
    for attempt in range(3):
        try:
            response = requests.get(
                f'{OPENDART_BASE}/document.xml',
                params={'crtfc_key': API_KEY, 'rcept_no': rcept_no},
                timeout=REQUEST_TIMEOUT,
            )
            response.raise_for_status()
            content = response.content
            if content[:2] != b'PK':
                snippet = content[:500].decode('utf-8', errors='replace')
                raise RuntimeError(f'document.xml 에러 응답: {snippet}')

            xml_files = []
            with zipfile.ZipFile(io.BytesIO(content)) as zip_file:
                for name in zip_file.namelist():
                    if not name.lower().endswith('.xml'):
                        continue
                    xml_files.append((name, decode_xml_bytes(zip_file.read(name))))
            if not xml_files:
                raise RuntimeError('document.xml ZIP 안에서 XML 파일을 찾지 못했습니다.')
            return xml_files
        except (requests.RequestException, zipfile.BadZipFile, RuntimeError) as exc:
            last_error = exc
            if attempt < 2:
                time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'document.xml 다운로드 실패: {last_error}')

## 6. II, IV, VI 섹션 추출

사업보고서 원문에서 `SECTION-1` 제목을 보고 II, IV, VI 섹션을 추출합니다. 이 세 섹션이 ESG 관련 문단 후보를 찾는 대상입니다.

In [6]:
TARGET_SECTION_LABELS = ('II', 'IV', 'VI')


def local_name(tag: str) -> str:
    """XML namespace를 제거한 태그명을 반환합니다."""
    return tag.rsplit('}', 1)[-1].upper()


def direct_title_text(element: ET.Element) -> str:
    """SECTION-1 바로 아래 TITLE 텍스트만 읽습니다."""
    for child in list(element):
        if local_name(child.tag) == 'TITLE':
            return ''.join(child.itertext()).strip()
    return ''


def normalize_roman_title(title: str) -> str:
    """로마 숫자 표기를 ASCII 표기로 맞춥니다."""
    return (
        title.replace('Ⅱ', 'II')
        .replace('Ⅳ', 'IV')
        .replace('Ⅵ', 'VI')
        .replace('ⅱ', 'II')
        .replace('ⅳ', 'IV')
        .replace('ⅵ', 'VI')
    )


def classify_section(title: str) -> str | None:
    """섹션 제목이 II, IV, VI 중 하나인지 판별합니다."""
    compact = re.sub(r'\s+', '', normalize_roman_title(title or ''))
    if re.match(r'^II\.?사업의?내용', compact):
        return 'II'
    if re.match(r'^IV\.?이사의?경영진단', compact):
        return 'IV'
    if re.match(r'^VI\.?이사회', compact):
        return 'VI'
    return None


def normalize_text(text: str) -> str:
    """HTML entity와 공백을 분석하기 쉬운 형태로 정리합니다."""
    text = unescape(text or '')
    text = re.sub(r'[ \t\r\f]+', ' ', text)
    text = re.sub(r'\n[ \t]*', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def strip_xml_tags(raw: str) -> str:
    """DART XML/HTML 조각을 plain text로 변환합니다."""
    raw = re.sub(r'</(P|TITLE|TR|LI|H[1-6]|TABLE|SECTION-[1-9]|PGBRK)>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'<BR\s*/?>', '\n', raw, flags=re.IGNORECASE)
    raw = re.sub(r'<[^>]+>', ' ', raw)
    return normalize_text(raw)


def element_to_text(element: ET.Element) -> str:
    """XML element를 plain text로 변환합니다."""
    raw = ET.tostring(element, encoding='unicode', method='xml')
    return strip_xml_tags(raw)


def parse_xml_root(xml_text: str) -> ET.Element | None:
    """파싱 가능한 XML root를 반환합니다."""
    try:
        return ET.fromstring(xml_text)
    except ET.ParseError:
        cleaned = re.sub(r'&(?!(?:amp|lt|gt|quot|apos|#\d+);)', '&amp;', xml_text)
        cleaned = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', cleaned)
        try:
            return ET.fromstring(cleaned)
        except ET.ParseError:
            return None


def title_from_section_opening(section_opening: str) -> str:
    """SECTION-1 시작부에서 TITLE 텍스트를 뽑습니다."""
    match = re.search(r'<TITLE\b[^>]*>(.*?)</TITLE>', section_opening, flags=re.IGNORECASE | re.DOTALL)
    if not match:
        return ''
    return strip_xml_tags(match.group(1))


def extract_sections_from_raw_xml_text(xml_text: str) -> dict[str, str | None]:
    """XML 파서가 실패하는 DART 원문에서 SECTION-1 문자열 기준으로 섹션을 추출합니다."""
    sections: dict[str, str | None] = {label: None for label in TARGET_SECTION_LABELS}
    section_starts = list(re.finditer(r'<SECTION-1\b', xml_text, flags=re.IGNORECASE))

    for idx, match in enumerate(section_starts):
        start = match.start()
        end = section_starts[idx + 1].start() if idx + 1 < len(section_starts) else len(xml_text)
        block = xml_text[start:end]
        label = classify_section(title_from_section_opening(block[:3000]))
        if label and not sections[label]:
            sections[label] = strip_xml_tags(block)

    return sections


def extract_sections_from_xml_files(xml_files: list[tuple[str, str]]) -> dict[str, str | None]:
    """여러 XML 파일에서 II, IV, VI 섹션 텍스트를 추출합니다."""
    sections: dict[str, str | None] = {label: None for label in TARGET_SECTION_LABELS}

    for _, xml_text in xml_files:
        root = parse_xml_root(xml_text)
        if root is not None:
            for element in root.iter():
                if local_name(element.tag) != 'SECTION-1':
                    continue
                label = classify_section(direct_title_text(element))
                if label and not sections[label]:
                    sections[label] = element_to_text(element)

        if not all(sections.values()):
            fallback_sections = extract_sections_from_raw_xml_text(xml_text)
            for label, section_text in fallback_sections.items():
                if section_text and not sections[label]:
                    sections[label] = section_text

        if all(sections.values()):
            break

    return sections

## 7. seed dictionary 기반 ESG passage 추출

`seed_dictionary.csv`의 `pattern`을 기준으로 II, IV, VI 섹션에서 ESG 관련 문단 후보를 찾습니다. 이 단계는 완성된 ESG 사전이 아니라, 원문 확인과 후속 전처리를 위한 재현 가능한 후보 추출입니다.

In [7]:
def load_seed_patterns(path: Path = SEED_DICTIONARY_PATH) -> list[dict[str, Any]]:
    """seed_dictionary.csv에서 dimension별 정규식 패턴을 읽습니다."""
    seeds = pd.read_csv(path)
    patterns = []
    for _, row in seeds.iterrows():
        pattern = str(row.get('pattern', '')).strip()
        if not pattern or pattern.lower() == 'nan':
            continue
        patterns.append(
            {
                'dimension': str(row.get('dimension', '')).strip(),
                'seed_term': str(row.get('seed_term', '')).strip(),
                'regex': re.compile(pattern, flags=re.IGNORECASE),
            }
        )
    return patterns


def split_passage_candidates(text: str) -> list[str]:
    """섹션 텍스트를 문단 후보로 나눕니다."""
    if not text:
        return []
    chunks = re.split(r'\n{2,}|(?<=[다음임함음요오])\.\s+', text)
    cleaned = []
    for chunk in chunks:
        passage = normalize_text(chunk)
        if 40 <= len(passage) <= 2000:
            cleaned.append(passage)
    return cleaned


def regex_matches(regex: re.Pattern, text: str) -> list[str]:
    """정규식 결과가 튜플이어도 문자열 목록으로 정규화합니다."""
    values = []
    for match in regex.findall(text):
        if isinstance(match, tuple):
            values.extend(str(part) for part in match if part)
        else:
            values.append(str(match))
    return values


def extract_esg_passages(
    sections: dict[str, str | None],
    seed_patterns: list[dict[str, Any]],
) -> list[dict[str, str]]:
    """II, IV, VI 섹션에서 seed pattern과 매칭되는 passage를 추출합니다."""
    passages = []
    seen = set()

    for section_label in TARGET_SECTION_LABELS:
        section_text = sections.get(section_label) or ''
        for passage in split_passage_candidates(section_text):
            matched_by_dimension: dict[str, set[str]] = {}
            for seed in seed_patterns:
                matches = sorted(set(regex_matches(seed['regex'], passage)))
                if not matches:
                    continue
                dimension = seed['dimension']
                matched_by_dimension.setdefault(dimension, set()).update(matches)
                matched_by_dimension[dimension].add(seed['seed_term'])

            for dimension, terms in matched_by_dimension.items():
                key = (section_label, dimension, passage[:250])
                if key in seen:
                    continue
                seen.add(key)
                passages.append(
                    {
                        'section': section_label,
                        'passage': passage,
                        'dimension': dimension,
                        'matched_terms': '|'.join(sorted(term for term in terms if term)),
                    }
                )

    return passages


seed_patterns = load_seed_patterns()
print(f'seed patterns: {len(seed_patterns)}')

seed patterns: 30


## 8. firm-year 한 행 처리 함수

한 행은 `corp_code` 확인, 사업보고서 선택, 원문 다운로드, 섹션 추출, ESG passage 추출 순서로 처리됩니다. 이미 정상 JSON이 있으면 API 호출을 생략할 수 있습니다.

In [8]:
CHECKPOINT_COLUMNS = [
    'company_name',
    'stock_code',
    'corp_code',
    'esg_year',
    'fiscal_year',
    'rcept_no',
    'rcept_dt',
    'viewer_url',
    'report_nm',
    'status',
    'failure_stage',
    'failure_reason',
    'section_II_chars',
    'section_IV_chars',
    'section_VI_chars',
    'passage_count',
]

PASSAGE_COLUMNS = [
    'company_name',
    'stock_code',
    'corp_code',
    'esg_year',
    'fiscal_year',
    'rcept_no',
    'rcept_dt',
    'viewer_url',
    'section',
    'passage',
    'dimension',
    'matched_terms',
]

SUCCESS_STATUSES = {'OK', 'OK_NO_PASSAGE', 'SKIP_EXISTING', 'SKIP_EXISTING_NO_PASSAGE'}


def report_json_path(stock_code: str, fiscal_year: int) -> Path:
    return REPORT_DIR / f'{stock_code}_{fiscal_year}.json'


def write_json_atomic(path: Path, payload: dict[str, Any]) -> None:
    """중간 실패로 깨진 JSON이 남지 않도록 임시 파일 후 교체합니다."""
    tmp_path = path.with_suffix(path.suffix + '.tmp')
    tmp_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    tmp_path.replace(path)


def load_existing_json(path: Path) -> dict[str, Any] | None:
    """기존 JSON 캐시를 읽습니다."""
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except json.JSONDecodeError:
        return None


def make_base_checkpoint(row: pd.Series) -> dict[str, Any]:
    return {
        'company_name': row.get('company_name', ''),
        'stock_code': str(row.get('stock_code', '')).zfill(6),
        'corp_code': '',
        'esg_year': int(row.get('esg_year')),
        'fiscal_year': int(row.get('fiscal_year')),
        'rcept_no': '',
        'rcept_dt': '',
        'viewer_url': '',
        'report_nm': '',
        'status': 'FAIL',
        'failure_stage': '',
        'failure_reason': '',
        'section_II_chars': 0,
        'section_IV_chars': 0,
        'section_VI_chars': 0,
        'passage_count': 0,
    }


def make_passage_rows(
    row: pd.Series,
    checkpoint: dict[str, Any],
    passages: list[dict[str, str]],
) -> list[dict[str, Any]]:
    """passage 단위 CSV 행을 만듭니다."""
    rows = []
    for passage in passages:
        rows.append(
            {
                'company_name': row.get('company_name', ''),
                'stock_code': checkpoint.get('stock_code', ''),
                'corp_code': checkpoint.get('corp_code', ''),
                'esg_year': checkpoint.get('esg_year'),
                'fiscal_year': checkpoint.get('fiscal_year'),
                'rcept_no': checkpoint.get('rcept_no', ''),
                'rcept_dt': checkpoint.get('rcept_dt', ''),
                'viewer_url': checkpoint.get('viewer_url', ''),
                'section': passage.get('section', ''),
                'passage': passage.get('passage', ''),
                'dimension': passage.get('dimension', ''),
                'matched_terms': passage.get('matched_terms', ''),
            }
        )
    return rows


def checkpoint_from_payload(row: pd.Series, payload: dict[str, Any], skipped: bool) -> dict[str, Any]:
    sections = payload.get('sections', {}) or {}
    passages = payload.get('passages', []) or []
    checkpoint = make_base_checkpoint(row)
    checkpoint.update(
        {
            'corp_code': payload.get('corp_code', ''),
            'rcept_no': payload.get('rcept_no', ''),
            'rcept_dt': payload.get('rcept_dt', ''),
            'viewer_url': payload.get('viewer_url', ''),
            'report_nm': payload.get('report_nm', ''),
            'status': 'SKIP_EXISTING' if skipped and passages else 'SKIP_EXISTING_NO_PASSAGE' if skipped else payload.get('status', 'OK'),
            'failure_stage': payload.get('failure_stage', ''),
            'failure_reason': payload.get('failure_reason', ''),
            'section_II_chars': len(sections.get('II') or ''),
            'section_IV_chars': len(sections.get('IV') or ''),
            'section_VI_chars': len(sections.get('VI') or ''),
            'passage_count': len(passages),
        }
    )
    return checkpoint


def process_firm_year(
    row: pd.Series,
    skip_existing: bool = True,
) -> tuple[dict[str, Any], list[dict[str, Any]]]:
    """company_master 한 행을 처리하고 checkpoint와 passage rows를 반환합니다."""
    checkpoint = make_base_checkpoint(row)
    stock_code = checkpoint['stock_code']
    fiscal_year = checkpoint['fiscal_year']
    output_path = report_json_path(stock_code, fiscal_year)

    if skip_existing:
        existing = load_existing_json(output_path)
        existing_sections = (existing or {}).get('sections', {}) or {}
        has_existing_sections = any(existing_sections.get(label) for label in TARGET_SECTION_LABELS)
        if existing and existing.get('status') in {'OK', 'OK_NO_PASSAGE'} and has_existing_sections:
            checkpoint = checkpoint_from_payload(row, existing, skipped=True)
            return checkpoint, make_passage_rows(row, checkpoint, existing.get('passages', []) or [])

    corp_code = stock_to_corp.get(stock_code)
    if not corp_code:
        checkpoint.update({'failure_stage': 'corp_code', 'failure_reason': 'stock_code에 대응하는 corp_code 없음'})
        write_json_atomic(output_path, {**checkpoint, 'sections': {}, 'passages': []})
        return checkpoint, []
    checkpoint['corp_code'] = corp_code

    try:
        selection = select_annual_report(corp_code, stock_code, fiscal_year)
    except Exception as exc:
        checkpoint.update({'failure_stage': 'list.json', 'failure_reason': str(exc)})
        write_json_atomic(output_path, {**checkpoint, 'sections': {}, 'passages': []})
        return checkpoint, []

    checkpoint.update(
        {
            'report_nm': selection.report_nm,
            'rcept_no': selection.rcept_no,
            'rcept_dt': selection.rcept_dt,
            'viewer_url': selection.viewer_url,
        }
    )

    try:
        xml_files = fetch_report_xml_files(selection.rcept_no)
    except Exception as exc:
        checkpoint.update({'failure_stage': 'document.xml', 'failure_reason': str(exc)})
        write_json_atomic(output_path, {**checkpoint, 'sections': {}, 'passages': []})
        return checkpoint, []

    sections = extract_sections_from_xml_files(xml_files)
    passages = extract_esg_passages(sections, seed_patterns)

    checkpoint.update(
        {
            'status': 'OK' if passages else 'OK_NO_PASSAGE',
            'section_II_chars': len(sections.get('II') or ''),
            'section_IV_chars': len(sections.get('IV') or ''),
            'section_VI_chars': len(sections.get('VI') or ''),
            'passage_count': len(passages),
        }
    )

    payload = {
        **checkpoint,
        'sections': sections,
        'passages': passages,
        'xml_file_names': [name for name, _ in xml_files],
    }
    write_json_atomic(output_path, payload)
    return checkpoint, make_passage_rows(row, checkpoint, passages)

## 9. 샘플 1-2개 실행

전체 381행을 실행하기 전에 삼성전자 같은 단건으로 API 키, 보고서 선택, 섹션 추출, passage 추출이 모두 작동하는지 확인합니다.

In [9]:
def run_sample(stock_code: str = '005930', fiscal_year: int = 2024) -> tuple[pd.DataFrame, pd.DataFrame]:
    """단건 테스트를 실행합니다."""
    target = company_master[
        (company_master['stock_code'] == stock_code.zfill(6))
        & (company_master['fiscal_year'] == fiscal_year)
    ]
    if target.empty:
        raise ValueError(f'company_master에 {stock_code}, {fiscal_year} 행이 없습니다.')

    checkpoint, passages = process_firm_year(target.iloc[0], skip_existing=False)
    checkpoint_df = pd.DataFrame([checkpoint], columns=CHECKPOINT_COLUMNS)
    passages_df = pd.DataFrame(passages, columns=PASSAGE_COLUMNS)

    print(checkpoint_df.T)
    print(f'passage 후보 수: {len(passages_df)}')
    return checkpoint_df, passages_df


# 기본 샘플 실행입니다. 네트워크 호출이 발생합니다.
sample_checkpoint, sample_passages = run_sample('005930', 2024)
sample_passages.head()

                                                                  0
company_name                                                   삼성전자
stock_code                                                   005930
corp_code                                                  00126380
esg_year                                                       2025
fiscal_year                                                    2024
rcept_no                                             20250311001085
rcept_dt                                                   20250311
viewer_url        https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
report_nm                                           사업보고서 (2024.12)
status                                                           OK
failure_stage                                                      
failure_reason                                                     
section_II_chars                                              38940
section_IV_chars                                

,company_name,stock_code,corp_code,esg_year,fiscal_year,rcept_no,rcept_dt,viewer_url,section,passage,dimension,matched_terms
0,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,당사의 자본관리 목적은 계속기업으로서 주주 및 이해관계자에게 이익을 지속적으로제공할...,G,주주
1,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,"그리고 당사는 레인보우로보틱스㈜ 및 그 최대주주 등과 체결한 주주간 계약에 따라,일...",G,주주
2,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,"또한 레인보우로보틱스㈜의 최대주주 등은 일정한 사유가 발생할 경우, 당사가 보유한 ...",G,주주
3,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,당사는 2024년 12월 31일 이사회에서 레인보우로보틱스㈜의 최대주주 등 6인이 ...,G,이사회|주주
4,삼성전자,005930,00126380,2025,2024,20250311001085,20250311,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...,II,"2024년말 현재 해당 콜옵션의 공정가치는 안진회계법인이 평가하였습니다.또한, 종속...",G,주주


## 10. 전체 381행 배치 실행

아래 셀을 실행하면 전체 입력 행을 순서대로 처리합니다. 성공, passage 미검출, 실패, 기존 파일 재사용 모두 `dart_collection_checkpoint.csv`에 기록합니다.

In [16]:
def collect_all(
    df: pd.DataFrame = company_master,
    start: int = 1,
    end: int | None = None,
    skip_existing: bool = True,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """company_master 구간을 처리하고 CSV 산출물을 저장합니다."""
    if end is None:
        end = len(df)
    target = df.iloc[start - 1:end]

    all_checkpoints: list[dict[str, Any]] = []
    all_passages: list[dict[str, Any]] = []

    print(f'수집 시작: {start}-{end}행, 총 {len(target)}건')
    started_at = time.time()

    for offset, (_, row) in enumerate(target.iterrows(), start=0):
        row_number = start + offset
        label = f'{row_number:03d}/{end:03d} {row["stock_code"]} {int(row["fiscal_year"])} {row.get("company_name", "")}'
        try:
            checkpoint, passages = process_firm_year(row, skip_existing=skip_existing)
        except Exception as exc:
            checkpoint = make_base_checkpoint(row)
            checkpoint.update({'failure_stage': 'unhandled', 'failure_reason': str(exc)})
            passages = []

        all_checkpoints.append(checkpoint)
        all_passages.extend(passages)
        print(f'{label}: {checkpoint["status"]}, passages={checkpoint["passage_count"]}, reason={checkpoint["failure_reason"]}')
        time.sleep(SLEEP_BETWEEN_CALLS)

    checkpoint_df = pd.DataFrame(all_checkpoints, columns=CHECKPOINT_COLUMNS)
    passages_df = pd.DataFrame(all_passages, columns=PASSAGE_COLUMNS)

    checkpoint_df.to_csv(CHECKPOINT_CSV_PATH, index=False, encoding='utf-8-sig')
    passages_df.to_csv(PASSAGES_CSV_PATH, index=False, encoding='utf-8-sig')

    elapsed = time.time() - started_at
    print(f'완료: {elapsed:.1f}초')
    print(checkpoint_df['status'].value_counts(dropna=False))
    print(f'저장: {CHECKPOINT_CSV_PATH}')
    print(f'저장: {PASSAGES_CSV_PATH}')
    return checkpoint_df, passages_df


# 전체 381행 수집입니다. 필요할 때 주석을 해제해 실행하세요.
checkpoint_df, passages_df = collect_all(skip_existing=True)

수집 시작: 1-381행, 총 381건
001/381 005930 2022 삼성전자: SKIP_EXISTING, passages=142, reason=
002/381 005930 2023 삼성전자: SKIP_EXISTING, passages=125, reason=
003/381 005930 2024 삼성전자: SKIP_EXISTING, passages=136, reason=
004/381 001460 2022 BYC: SKIP_EXISTING, passages=43, reason=
005/381 001460 2023 BYC: SKIP_EXISTING, passages=47, reason=
006/381 001460 2024 BYC: SKIP_EXISTING, passages=63, reason=
007/381 005380 2022 현대자동차: SKIP_EXISTING, passages=183, reason=
008/381 005380 2023 현대자동차: SKIP_EXISTING, passages=198, reason=
009/381 005380 2024 현대자동차: SKIP_EXISTING, passages=225, reason=
010/381 000120 2022 CJ대한통운: SKIP_EXISTING, passages=94, reason=
011/381 000120 2023 CJ대한통운: SKIP_EXISTING, passages=107, reason=
012/381 000120 2024 CJ대한통운: SKIP_EXISTING, passages=105, reason=
013/381 097950 2022 CJ제일제당: SKIP_EXISTING, passages=116, reason=
014/381 097950 2023 CJ제일제당: SKIP_EXISTING, passages=120, reason=
015/381 097950 2024 CJ제일제당: SKIP_EXISTING, passages=115, reason=
016/381 001040 2022 CJ: S

## 11. 체크포인트 CSV 검증 및 실패 사유 요약

수집 후에는 성공 행의 식별자와 섹션 길이를 확인하고, 실패 행은 최종 보고서에 설명할 수 있도록 단계별로 요약합니다.

In [17]:
def validate_collection_outputs(
    checkpoint_path: Path = CHECKPOINT_CSV_PATH,
    passages_path: Path = PASSAGES_CSV_PATH,
) -> dict[str, Any]:
    """저장된 수집 산출물을 점검합니다."""
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'체크포인트 CSV가 없습니다: {checkpoint_path}')

    checkpoint_df = pd.read_csv(
        checkpoint_path,
        dtype={'stock_code': 'string', 'corp_code': 'string', 'rcept_no': 'string'},
    )
    if passages_path.exists():
        passages_df = pd.read_csv(
            passages_path,
            dtype={'stock_code': 'string', 'corp_code': 'string', 'rcept_no': 'string'},
        )
    else:
        passages_df = pd.DataFrame(columns=PASSAGE_COLUMNS)

    success_mask = checkpoint_df['status'].isin(SUCCESS_STATUSES)
    success_df = checkpoint_df[success_mask]

    issues = []
    if len(checkpoint_df) != 381:
        issues.append(f'체크포인트 CSV 행 수가 381이 아닙니다: {len(checkpoint_df)}')

    expected_columns = CHECKPOINT_COLUMNS
    missing_columns = [col for col in expected_columns if col not in checkpoint_df.columns]
    if missing_columns:
        issues.append(f'체크포인트 CSV 필수 컬럼 누락: {missing_columns}')

    for col in ['corp_code', 'rcept_no', 'rcept_dt', 'viewer_url']:
        missing = int(success_df[col].isna().sum() + (success_df[col].fillna('') == '').sum())
        if missing:
            issues.append(f'성공 행 중 {col} 누락: {missing}')

    failure_df = checkpoint_df[~success_mask]
    missing_failure_reason = int((failure_df['failure_reason'].fillna('') == '').sum()) if len(failure_df) else 0
    if missing_failure_reason:
        issues.append(f'실패 행 중 failure_reason 누락: {missing_failure_reason}')

    failure_summary = (
        failure_df
        .groupby(['failure_stage', 'failure_reason'], dropna=False)
        .size()
        .reset_index(name='count')
        .sort_values('count', ascending=False)
    )

    summary = {
        'checkpoint_rows': len(checkpoint_df),
        'passage_rows': len(passages_df),
        'success_rows': int(success_mask.sum()),
        'failure_rows': int((~success_mask).sum()),
        'issues': issues,
    }

    print(json.dumps(summary, ensure_ascii=False, indent=2))
    if len(failure_summary):
        display(failure_summary.head(30))
    return summary



In [18]:
output_validation = validate_collection_outputs()

{
  "checkpoint_rows": 381,
  "passage_rows": 39322,
  "success_rows": 376,
  "failure_rows": 5,
  "issues": []
}


,failure_stage,failure_reason,count
0,list.json,2022 회계연도 사업보고서를 찾지 못했습니다. 조회 예시: ['사업보고서 (202...,1
1,list.json,2022 회계연도 사업보고서를 찾지 못했습니다. 조회 예시: ['사업보고서 (202...,1
2,list.json,2023 회계연도 사업보고서를 찾지 못했습니다. 조회 예시: ['사업보고서 (202...,1
3,list.json,2023 회계연도 사업보고서를 찾지 못했습니다. 조회 예시: ['사업보고서 (202...,1
4,list.json,2024 회계연도 사업보고서를 찾지 못했습니다. 조회 예시: ['[기재정정]사업보고...,1
